In [0]:
!pip install feature_engine

In [0]:
import mlflow

model = mlflow.sklearn.load_model("models:/workspace.analytics.modelo_asn_tmw/2")
model

In [0]:
%sql
select max(dtRef) from workspace.analytics.fs_pontos

In [0]:
df = spark.sql(""" 
    SELECT *
            EXCEPT (t2.dtRef, t2.idCliente)
    FROM workspace.analytics.fs_pontos AS t1

    LEFT JOIN workspace.analytics.fs_cursos AS t2
    ON t1.IdCliente = t2.idCliente
    AND t1.dtRef = t2.dtRef

    WHERE t1.dtRef = (select max(dtRef) from workspace.analytics.fs_pontos)

""").toPandas()

In [0]:
X = df[model.feature_names_in_]
predict = model.predict_proba(X)[:, 0]

df['proba'] = predict
df_list_50 = df[['IdCliente', 'proba']].head(50).sort_values('proba', ascending = False)
df_list_50.reset_index(drop = True)
df_list_50.index = range(1, len(df_list_50) + 1)

In [0]:
df.head()